In [ ]:
EXP_NAME = "prev40_dim6_cont_FULL_10"
TRAINING_DATASET = "synth/prev40_dim6_cont/train.csv"
TEST_DATASET = "synth/prev40_dim6_cont/test.csv"
FEATURE_MAP = "scm/synth/toy_6_10.json"
SEED = 4

GROUP_LABELS = ["1", "0"]

class ModelLabels:
  baseline = 'Baseline (Full)'
  baseline_unaware = 'Baseline (S-blind)'
  ablation = 'Baseline (Xbio & Xind only)'
  fair = 'Fair (S-aware)'
  fair_unaware = 'Fair (S-blind)'

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'
  if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
from notebooks.analysis_utils import mutual_info_grouped, plot_cat_feature, plot_cont_feature, plot_cf_cont_feature_comparison, plot_cf_cat_feature_comparison, CustPalette

from src.config import Config

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1)

In [ ]:
test_latents = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{EXP_NAME}/test_latent_space.csv')
test_counterfactuals = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{EXP_NAME}/test_counterfactuals.csv')
training_latents = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{EXP_NAME}/train_latent_space.csv')
training_counterfactuals = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{EXP_NAME}/train_counterfactuals.csv')
test_dataset = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{TEST_DATASET}')
training_dataset = pd.read_csv(f'{PROJECT_ROOT}/{Config.DATA_DIR}/{TRAINING_DATASET}')

classifiers_results = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/bootstrapped_evaluation_results.csv')
classifiers_results['model'] = classifiers_results['model'].replace({
  'baseline': ModelLabels.baseline,
  'baseline_unaware': ModelLabels.baseline_unaware,
  'ablation': ModelLabels.ablation,
  'fair': ModelLabels.fair,
  'fair_unaware': ModelLabels.fair_unaware
})
classifiers_results.sort_values(by='model', axis=0, inplace=True)

with open(f"{PROJECT_ROOT}/configs/{FEATURE_MAP}", 'r') as f:
  feature_map = json.load(f)

In [ ]:
training_df = training_latents.merge(training_dataset, how="inner", left_on="patient_index", right_index=True)
training_df = training_df.merge(training_counterfactuals, how="inner", on="patient_index", suffixes=("", "_cf"))
training_df.drop('sample_index_cf', axis=1, inplace=True)

test_df = test_latents.merge(test_dataset, how="inner", left_on="patient_index", right_index=True)
test_df = test_df.merge(test_counterfactuals, how="inner", on="patient_index", suffixes=("", "_cf"))
test_df.drop('sample_index_cf', axis=1, inplace=True)

In [ ]:
desc_cols = [f['name'] for f in feature_map['desc']]
desc_cf_cols = [f"{f['name']}_cf" for f in feature_map['desc']]
latent_cols = [col for col in training_latents.columns if 'u_desc_' in col]
sens_col = feature_map['sens'][0]['name']
target_col = feature_map['target']['name']

test_prevalence = test_df[target_col].sum(axis=0) / len(test_df)

In [ ]:
from scipy.stats import sem, t
def get_95_ci(data):
  '''
    Calculates the 95% confidence interval for a given set of data.

    Inputs
      data: data as a Pandas Series

    Outputs
      interval: Array of the lower and upper bounds of the confidence interval
  '''
  n = len(data)
  mean = data.mean()
  std_err = sem(data, nan_policy='omit')
  if std_err == 0:
    return (mean, mean)
  interval = t.interval(0.95, n - 1, loc=mean, scale=std_err)
  return interval

def format_avg(metric, precision=3, perc=False):
  ci = get_95_ci(metric)
  if perc:
    return f"{round(metric.mean()*100, precision)}% ({round(ci[0]*100, precision)}%-{round(ci[1]*100, precision)}%)"
  else:
    return f"{round(metric.mean(), precision)} ({round(ci[0], precision)}-{round(ci[1], precision)})"

# Classifier results

In [ ]:
classifiers_results['f1'] = 2*classifiers_results['recall']*classifiers_results['ppv'] / (classifiers_results['recall'] + classifiers_results['ppv'])

## Global results

In [ ]:
global_results = classifiers_results[classifiers_results['subgroup'] == "Global"]
global_aggregated_results = global_results.groupby('model').agg({
  "auprc":( lambda x: format_avg(x, precision=2, perc=True)),
  "brier_score":format_avg,
  "recall":( lambda x: format_avg(x, precision=2, perc=True)),
  "ppv":( lambda x: format_avg(x, precision=2, perc=True)),
  "f1":( lambda x: format_avg(x, precision=2, perc=True)),
  "fnr":( lambda x: format_avg(x, precision=2, perc=True)),
  "fpr":( lambda x: format_avg(x, precision=2, perc=True)),
  "cf_harm_balanced":format_avg,
  "cf_harm_pos":format_avg,
  "cf_harm_neg":format_avg,
})

print(global_aggregated_results.reset_index().to_markdown(index=False))

In [ ]:
fig = plt.figure(figsize=(12, 5))

ax1 = fig.add_subplot([0, 0, 0.18, 0.8])
ax2 = fig.add_subplot([0.28, 0, 0.18, 0.8], sharey=ax1)
ax3 = fig.add_subplot([0.56, 0, 0.18, 0.8], sharey=ax1)
axes = [ax1,ax2,ax3]


custom_colors = {
  ModelLabels.baseline: CustPalette.secondary,
  ModelLabels.baseline_unaware: CustPalette.secondary_light,
  ModelLabels.ablation: CustPalette.secondary_x_light,
  ModelLabels.fair: CustPalette.primary,
  ModelLabels.fair_unaware: CustPalette.primary_light
}

sns.barplot(global_results, y='recall', hue="model", hue_order=list(custom_colors.keys()), gap=0.1, palette=custom_colors, errorbar=("ci", 95), ax=axes[0], legend=False)
sns.barplot(global_results, y='ppv', hue="model", hue_order=list(custom_colors.keys()), gap=0.1, palette=custom_colors, errorbar=("ci", 95), ax=axes[1], legend=False)
sns.barplot(global_results, y='auprc', hue="model", hue_order=list(custom_colors.keys()), gap=0.1, palette=custom_colors, errorbar=("ci", 95), ax=axes[2], legend=True)
axes[2].axhline(y=test_prevalence, color=CustPalette.tertiary, linewidth=4, label="Outcome prevalence")

metrics = ["Avg Global Recall", "Avg Global Precision", "Avg Global AUPRC"]
for i, ax in enumerate(axes):
  ax.set_ylabel("")
  ax.set_xlabel("")
  ax.set_ylim(0,1)
  ax.set_title(metrics[i], fontsize=18, pad=15, weight='bold')
  ax.tick_params(axis='both', labelsize=11)
  ax.set_xticks([])

  ax.grid(False)

  for spine in ['top', 'right', 'bottom']:
    ax.spines[spine].set_visible(False)

  if i == 0:
    ax.tick_params(axis='y', which='both', left=True, labelleft=True, labelsize=12)
  else:
    ax.tick_params(axis='y', which='both', left=True, labelleft=False)

fig.subplots_adjust(left=0.18, right=0.78, bottom=0.15, top=0.85, wspace=0.25)

leg = axes[2].legend(title="Classifier", title_fontsize=16, fontsize=16, loc='upper left', bbox_to_anchor=(1.05, 1), frameon=False)
leg._legend_box.align = "left"

plt.savefig(f"{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/global_results_poster.svg", format='svg', bbox_inches='tight')
# plt.savefig(f"{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/global_results_poster.png", dpi=300, bbox_inches='tight')
plt.show()

## Stratified results

In [ ]:
group_results = classifiers_results[classifiers_results['subgroup'] != "Global"].pivot_table(index=['model', 'bootstrap_idx'], columns='subgroup', values=['auprc', 'recall', 'ppv', 'f1', 'fpr'], aggfunc="mean")

# Add equalised odds to the results
lowest_tpr = group_results[[('recall','Group 0'), ('recall','Group 1')]].min(axis=1)
highest_tpr = group_results[[('recall','Group 0'), ('recall','Group 1')]].max(axis=1)
tpr_ratio = lowest_tpr / highest_tpr 
lowest_fpr = group_results[[('fpr','Group 0'), ('fpr','Group 1')]].min(axis=1)
highest_fpr = group_results[[('fpr','Group 0'), ('fpr','Group 1')]].max(axis=1)
fpr_ratio = lowest_fpr / highest_fpr 
group_results['equalised odds ratio'] = np.minimum(tpr_ratio, fpr_ratio)

In [ ]:
aggregated_group_results = group_results.groupby('model').agg(lambda x: format_avg(x, precision=2, perc=True))

aggregated_group_results.columns = [
    f"{metric} - {subgroup}" if pd.notna(subgroup) and subgroup != '' else f"{metric}"
    for metric, subgroup in aggregated_group_results.columns
]
print(aggregated_group_results.to_markdown())

In [ ]:
import textwrap

model_order = [
    ModelLabels.baseline,
    ModelLabels.baseline_unaware,
    ModelLabels.ablation,
    ModelLabels.fair,
    ModelLabels.fair_unaware
]

wrapped_labels = [textwrap.fill(label, width=14) for label in model_order]

metrics = ['recall', 'ppv']
titles = ["Avg Group Recall", "Avg Group Precision"]
y_positions = np.flip(np.arange(len(model_order)))

model_names = group_results.index.get_level_values(0)
pivoted = group_results.groupby(model_names).mean()
pivoted = pivoted.reindex(model_order)

fig = plt.figure(figsize=(12, 5))
ax1 = fig.add_subplot([0, 0, 0.32, 0.8])
ax2 = fig.add_subplot([0.48, 0, 0.32, 0.8])
axes = [ax1, ax2]

for i, ax in enumerate(axes):
    metric = metrics[i]
    metric_data = pivoted[metric]
    
    g0_values = metric_data['Group 0'].values
    g1_values = metric_data['Group 1'].values
    
    ax.hlines(
        y=y_positions, 
        xmin=np.minimum(g0_values, g1_values), 
        xmax=np.maximum(g0_values, g1_values), 
        color=CustPalette.grey_light,
        linewidth=4, 
        zorder=1
    )
    
    ax.scatter(
        g1_values, y_positions, 
        color=CustPalette.primary, 
        s=180, edgecolors='black', linewidths=0.5,
        label='Group 1' if i == 1 else "", 
        zorder=2
    )
    
    ax.scatter(
        g0_values, y_positions, 
        color=CustPalette.secondary, 
        s=180, edgecolors='black', linewidths=0.5,
        label='Group 0' if i == 1 else "", 
        zorder=2
    )
    
    ax.set_title(titles[i], fontsize=18, pad=15, weight='bold')
    ax.set_xlim(0, 1.1)
    
    ax.set_yticks([])
    ax.tick_params(axis='x', labelsize=14)
        
    ax.grid(False)
    ax.grid(axis='x', linestyle=':', alpha=0.6)
    
    for spine in ['top', 'right', 'left', 'bottom']:
        ax.spines[spine].set_visible(False)

axes[0].tick_params(axis='y', which='both', right=False, labelright=True, left=False, labelleft=False)
axes[0].set_yticks(y_positions)
axes[0].set_yticklabels(wrapped_labels, fontsize=16, ha='center', va='center')
axes[0].get_yaxis().set_tick_params(pad=60)

handles, _ = axes[1].get_legend_handles_labels()
axes[1].legend(
    handles=handles,
    loc="upper left", 
    bbox_to_anchor=(1, 1), 
    frameon=False, 
    fontsize=16,
    handletextpad=0.3,
    labels=GROUP_LABELS
)

plt.savefig(f"{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/group_results_dumbbell.svg", format='svg', bbox_inches='tight')
# plt.savefig(f"{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/group_results_dumbbell.png", dpi=300, bbox_inches='tight')
plt.show()

# Latent space

## Mutual Information with $X_{sens}$

### Training dataset

In [ ]:
mutual_info_grouped(
  training_df,
  feature_cols= desc_cols + latent_cols,
  target_col=sens_col,
  target_label="sensitive attribute",
  iterations=100,
  n_samples=400,
  seed=SEED
)

### Test dataset

In [ ]:
mutual_info_grouped(
  test_df,
  feature_cols= desc_cols + latent_cols,
  target_col=sens_col,
  target_label="sensitive attribute",
  iterations=100,
  n_samples=100,
  seed=SEED
)

# Counterfactuals

In [ ]:
for feature in feature_map['desc']:
  col_name = feature['name']
  cf_col_name = feature['name'] + "_cf"
  group_0_mask = training_df[sens_col] == 0
  group_1_mask = training_df[sens_col] == 1
  if feature['type'] == "continuous":
    plot_cf_cont_feature_comparison(training_df, col_name, cf_col_name, sens_col, target_col)
  else:
    plot_cf_cat_feature_comparison(training_df, col_name, cf_col_name, sens_col, target_col)